In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive/빅데이터의사결정분석/Project

/content/drive/MyDrive/빅데이터의사결정분석/Project


In [3]:
!pip install -q contractions

# **라이브러리 호출**

In [4]:
# 기본 라이브러리
import pandas as pd
import numpy as np
import tensorflow as tf

# 작업 진행률 시각화
from tqdm import tqdm
tqdm.pandas()

# NLTK 라이브러리
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# 텍스트 데이터 전처리
import re
import contractions
from collections import Counter
from bs4 import BeautifulSoup
from nltk.tokenize import word_tokenize

# Word2vec 텍스트 임베딩 모델
from gensim.models import Word2Vec

# 딥러닝 모델 라이브러리
from tensorflow import keras
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import  Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, SimpleRNN, LSTM, Bidirectional, MaxPooling1D, GlobalMaxPooling1D, Dense, concatenate, Dropout, Layer

# 머신러닝 모델 및 평가
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# 머신러닝 변수 추출
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

# **Utils**

In [10]:
def preprocess_text_lemmatized(text, normalization_dict=None):

    '''
    1. HTML 태그 제거
    2. 특정 단어 정규화 (옵션)
    3. 특수 문자 제거
    4. 모든 문자를 소문자로 변환
    5. 텍스트를 토큰화 및 불용어 제거
    6. 표제어 추출 및 원형 복원
    7. 표제어 추출 리스트를 String 형식으로 변경
    '''

  # HTML 태그 제거
    text_no_html = BeautifulSoup(text, "html.parser").get_text()

  # 특정 단어 정규화 (옵션)
    if normalization_dict:
        for old, new in normalization_dict.items():
            text_no_html = text_no_html.replace(old, new)

  # 특수 문자 제거
    text_no_specials = re.sub(r'[^a-zA-Z]', ' ', text_no_html)

  # 여러 공백을 하나로 축소 및 공백 정리
  # 띄어쓰기가 여럿 있으면, 띄어쓰기 하나로 만들어줌
    text_clean = re.sub(r'\s+', ' ', text_no_specials).strip()

  # 모든 문자를 소문자로 변환
    texts_lower = text_clean.lower()

  # 텍스트를 토큰화 및 불용어 제거
    tokens = word_tokenize(texts_lower)
    filtered_tokens = [word for word in tokens if word not in stopwords.words('english')]

  # lemmatizer Class 호출
    lemmatizer = WordNetLemmatizer()

  # 리스트 형태의 출력을 DataFrame에 저장할 수 있도록 String으로 변환해줌
    lemmatized_tokens = ' '.join([lemmatizer.lemmatize(word) for word in filtered_tokens])

    return lemmatized_tokens

In [11]:
def filter_text_by_frequency(df, column_name, min_frequency, out_column_name):

    '''
    1. 데이터프레임의 모든 텍스트를 하나의 문자열로 합치기
    2. 토큰화하여 단어의 빈도 계산하기
    3. 특정 빈도 이하의 단어 제거하기
    4. 필터링된 단어들로 새로운 텍스트 집합 생성
    5. 필터링된 텍스트를 새로운 데이터프레임 컬럼에 저장
    '''

  # 1. 데이터프레임의 모든 텍스트를 하나의 문자열로 합치기
    all_text = ' '.join(df[column_name])

  # 2. 토큰화하여 단어의 빈도 계산하기
    tokens = word_tokenize(all_text)
    word_counts = Counter(tokens)

  # 3. 특정 빈도 이하의 단어 제거하기
    filtered_words = [word for word, count in word_counts.items() if count > min_frequency]

  # 4. 필터링된 단어들로 새로운 텍스트 집합 생성
    filtered_texts = [' '.join(word for word in word_tokenize(text) if word in filtered_words) for text in df[column_name]]

  # 필터링된 텍스트를 새로운 데이터프레임 컬럼에 저장
    df[out_column_name] = filtered_texts

    return df

In [12]:
def tokenize_and_create_embedding_matrix(train_text, test_text, max_words=10000, max_sequence_length=300):
    '''
    1. 토크나이저 초기화 (OOV 토큰 포함)
    2. 훈련 데이터에 토크나이저를 적용해 학습
    3. 텍스트 데이터를 시퀀스로 변환
    4. 시퀀스 패딩 처리 (post 방식 사용)
    5. Word2Vec 모델로 텍스트간의 관계를 학습
    '''
  # 토크나이저 초기화 (OOV 토큰 포함)
    tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')

  # 훈련 데이터에 토크나이저를 적용해 학습
    tokenizer.fit_on_texts(train_text)

  # 텍스트 데이터를 시퀀스로 변환
    train_sequences = tokenizer.texts_to_sequences(train_text)
    test_sequences = tokenizer.texts_to_sequences(test_text)

  # 시퀀스 패딩 처리 (post 방식 사용)
    padded_train_text = pad_sequences(train_sequences, maxlen=max_sequence_length, padding='pre')
    padded_test_text = pad_sequences(test_sequences, maxlen=max_sequence_length, padding='pre')

  # 단어 인덱스 맵 생성 및 임베딩 행렬 준비
  # 임베딩 차원을 300으로 가정
    word_index = tokenizer.word_index
    total_words = min(max_words, len(word_index))
    embedding_matrix = np.zeros((total_words + 1, 300))

  # Word2Vec 모델에서 단어에 해당하는 벡터를 임베딩 행렬에 할당
    tokenized_x_train = train_text.apply(word_tokenize)
    tokenized_x_test = test_text.apply(word_tokenize)

  # Word2Vec 모델 훈련
    w2v = Word2Vec(sentences= tokenized_x_train, vector_size=300, window=5, min_count=1, workers=4)

  # 임베딩 행렬의 크기 출력
    for word, i in word_index.items():
        if i > max_words:
            continue
        if word in w2v.wv:
            embedding_matrix[i] = w2v.wv[word]

  # 패딩 처리된 텍스트 데이터와 임베딩 행렬, 사용된 총 단어 수, 시퀀스 최대 길이 반환
    return padded_train_text, padded_test_text, embedding_matrix, total_words, max_sequence_length


In [13]:
def CNN_LSTM_helpful(text_input, num_input):

    '''
    1. 모형의 입력은 텍스트, 숫자 변수(평점 포함) 2개로 구성됨
    2. 텍스트의 경우 CNN-LSTM을 거침
    3. 평점은 필수로 투입되고 숫자 변수는 Feature를 선택해서 투입
    5. 결과로 분류 확률값 출력
    '''

  # 텍스트 입력 레이어
    text_input = Input(shape=(300, ), dtype='float32', name='text_input')

  # 평점, 숫자 column 입력 레이어
  # 둘을 Concat 하여 투입함
    num_input = Input(shape=(5,), dtype='float32', name='num_input')

  # 텍스트 임베딩 레이어
    embedding_layer = Embedding(input_dim=total_words+1, output_dim=300,  embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
                              input_length=max_sequence_length, trainable=True, name='Embedding_Layer')(text_input)

  # 텍스트 처리 모델
  # 컨볼루션 레이어 및 맥스 풀링 레이어
    conv = Conv1D(filters=100, kernel_size=3, padding='causal', activation='relu', name='Conv1D_3')(embedding_layer)
    conv = MaxPooling1D(name='MaxPooling1D_3')(conv)

  # LSTM 레이어 추가
    lstm_layer = Bidirectional(LSTM(64, return_sequences=False, name='Bidirectional_LSTM'), name='Bidirectional')(conv)

  # MLP층 통과
    text_mlp_layer = Dense(112, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_text_1')(lstm_layer)
    text_mlp_layer = Dense(96, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_text_2')(text_mlp_layer)
    text_mlp_layer = Dense(80, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_text_3')(text_mlp_layer)

  # 숫자변수 처리 Layer
  # 숫자 Feature MLP 레이어에 정규화 시행
    num_mlp_layer = Dense(5, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_num_1')(num_input)
    num_mlp_layer = Dense(3, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_num_2')(num_mlp_layer)

  # 전체 벡터 합치기
    concatenated = concatenate([text_mlp_layer, num_mlp_layer])

  # 최종 MLP 레이어에 정규화 시행
    final_mlp = Dense(80, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_final_1')(concatenated)
    final_mlp = Dense(72, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_final_2')(final_mlp)
    final_mlp = Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_final_3')(final_mlp)

  # 출력 레이어
    output_layer = Dense(1, activation='sigmoid', name='Output_Layer')(final_mlp)

  # 모델 정의
    model = Model(inputs=[text_input, num_input], outputs=output_layer)

    return model

In [14]:
def CNN_helpful(text_input, num_input):

    '''
    1. 모형의 입력은 텍스트, 숫자 변수(평점 포함) 2개로 구성됨
    2. 텍스트의 경우 CNN만 거침
    3. 평점은 필수로 투입되고 숫자 변수는 Feature를 선택해서 투입
    5. 결과로 분류 확률값 출력
    '''

  # 텍스트 입력 레이어
    text_input = Input(shape=(300, ), dtype='float32', name='text_input')

  # 평점, 숫자 column 입력 레이어
  # 둘을 Concat 하여 투입함
    num_input = Input(shape=(5,), dtype='float32', name='num_input')

  # 텍스트 임베딩 레이어
    embedding_layer = Embedding(input_dim=total_words+1, output_dim=300,  embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
                              input_length=max_sequence_length, trainable=True, name='Embedding_Layer')(text_input)

  # 텍스트 처리 모델
  # 컨볼루션 레이어 및 글로벌 맥스 풀링 레이어
    conv = Conv1D(filters=100, kernel_size=3, padding='causal', activation='relu', name='Conv1D_3')(embedding_layer)
    conv = GlobalMaxPooling1D(name='GlobalMaxPooling1D_3')(conv)

  # MLP층 통과
    text_mlp_layer = Dense(100, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_text_1')(conv)
    text_mlp_layer = Dense(90, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_text_2')(text_mlp_layer)
    text_mlp_layer = Dense(80, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_text_3')(text_mlp_layer)

  # 숫자변수 처리 Layer
  # 숫자 Feature MLP 레이어에 정규화 시행
    num_mlp_layer = Dense(5, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_num_1')(num_input)
    num_mlp_layer = Dense(3, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_num_2')(num_mlp_layer)

  # 전체 벡터 합치기
    concatenated = concatenate([text_mlp_layer, num_mlp_layer])

  # 최종 MLP 레이어에 정규화 시행
    final_mlp = Dense(80, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_final_1')(concatenated)
    final_mlp = Dense(72, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_final_2')(final_mlp)
    final_mlp = Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_final_3')(final_mlp)

  # 출력 레이어
    output_layer = Dense(1, activation='sigmoid', name='Output_Layer')(final_mlp)

  # 모델 정의
    model = Model(inputs=[text_input, num_input], outputs=output_layer)

    return model

In [15]:
def CNN_RNN_helpful(text_input, num_input):

    '''
    1. 모형의 입력은 텍스트, 숫자 변수(평점 포함) 2개로 구성됨
    2. 텍스트의 경우 CNN-RNN을 거침
    3. 평점은 필수로 투입되고 숫자 변수는 Feature를 선택해서 투입
    5. 결과로 분류 확률값 출력
    '''

  # 텍스트 입력 레이어
    text_input = Input(shape=(300, ), dtype='float32', name='text_input')

  # 평점, 숫자 column 입력 레이어
  # 둘을 Concat 하여 투입함
    num_input = Input(shape=(5,), dtype='float32', name='num_input')

  # 텍스트 임베딩 레이어
    embedding_layer = Embedding(input_dim=total_words+1, output_dim=300,  embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
                              input_length=max_sequence_length, trainable=True, name='Embedding_Layer')(text_input)

  # 텍스트 처리 모델
  # 컨볼루션 레이어 및 맥스 풀링 레이어
    conv = Conv1D(filters=100, kernel_size=3, padding='causal', activation='relu', name='Conv1D_3')(embedding_layer)
    conv = MaxPooling1D(name='MaxPooling1D_3')(conv)

  # RNN 레이어 추가
    RNN_layer = Bidirectional(SimpleRNN(64, return_sequences=False, name='Simple_RNN'))(conv)

  # MLP층 통과
    text_mlp_layer = Dense(112, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_text_1')(RNN_layer)
    text_mlp_layer = Dense(96, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_text_2')(text_mlp_layer)
    text_mlp_layer = Dense(80, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_text_3')(text_mlp_layer)

  # 숫자변수 처리 Layer
  # 숫자 Feature MLP 레이어에 정규화 시행
    num_mlp_layer = Dense(5, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_num_1')(num_input)
    num_mlp_layer = Dense(3, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_num_2')(num_mlp_layer)

  # 전체 벡터 합치기
    concatenated = concatenate([text_mlp_layer, num_mlp_layer])

  # 최종 MLP 레이어에 정규화 시행
    final_mlp = Dense(80, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_final_1')(concatenated)
    final_mlp = Dense(72, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_final_2')(final_mlp)
    final_mlp = Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.01), name=f'MLP_Layer_final_3')(final_mlp)

  # 출력 레이어
    output_layer = Dense(1, activation='sigmoid', name='Output_Layer')(final_mlp)

  # 모델 정의
    model = Model(inputs=[text_input, num_input], outputs=output_layer)

    return model

# **데이터 불러오기**
- 원본 데이터에 Label 불균형 문제가 있어서, 정상리뷰와 Fake 리뷰의 개수를 맞춰줌

In [16]:
review = pd.read_csv('project_review_helpful.csv')

In [17]:
print(review.shape)
review.head()

(24838, 21)


In [18]:
# 평점 범위 확인
review['rating'].value_counts()

rating
5    15014
4     5833
3     1893
1     1211
2      887
Name: count, dtype: int64

In [19]:
# fake label 개수 확인
review['fake'].value_counts()

fake
0    12419
1    12419
Name: count, dtype: int64

In [20]:
# 진짜 리뷰만 리뷰 작성자가 명시되어 있음
review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24838 entries, 0 to 24837
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   rating                    24838 non-null  int64  
 1   reviewer_url              12419 non-null  object 
 2   review_text               24838 non-null  object 
 3   photo                     24838 non-null  int64  
 4   location                  24838 non-null  int64  
 5   depth                     24838 non-null  int64  
 6   text_structure            24838 non-null  float64
 7   Cognition                 24838 non-null  float64
 8   Affect                    24838 non-null  float64
 9   emotion                   24838 non-null  float64
 10  Social                    24838 non-null  float64
 11  Perception                24838 non-null  float64
 12  motion                    24838 non-null  float64
 13  Psychological Distancing  24838 non-null  float64
 14  fake  

# **텍스트 전처리**

In [21]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [22]:
# 전처리 함수 적용
review['lemmatized_text'] = review['review_text'].progress_apply(preprocess_text_lemmatized)
review.head()

  0%|          | 15/24838 [00:01<30:33, 13.54it/s] <ipython-input-10-cbb3593b54f6>:14: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text_no_html = BeautifulSoup(text, "html.parser").get_text()
100%|██████████| 24838/24838 [02:49<00:00, 146.50it/s]


In [23]:
# stemmting, lemmatize 한 단어들 중에서 5개 이하로 언급된 단어 제외
filter_text_by_frequency(review, 'lemmatized_text', min_frequency=5, out_column_name='filter_lemmatized_text')

# **Train, Test 데이터 분할**

In [29]:
# 훈련 및 테스트 데이터 분할
train, test = train_test_split(review, test_size=0.2, random_state=42,  stratify= review["fake"])

train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

# 훈련 데이터 리뷰 텍스트와 라벨 추출
x_train = train['filter_lemmatized_text']
y_train = np.array(train['fake'])

# 테스트 데이터 리뷰 텍스트와 라벨 추출
x_test = test['filter_lemmatized_text']
y_test = np.array(test['fake'])

In [30]:
padded_train_text, padded_test_text, embedding_matrix, total_words, max_sequence_length = tokenize_and_create_embedding_matrix(x_train, x_test, max_words=10000, max_sequence_length=300)

# **숫자 데이터 Column 정보**

* **depth (깊이):**
  - 리뷰 텍스트의 길이나 복잡성을 나타내는 지표

* **Cognition:**
  - 리뷰에서 인지적 단어의 비율​

* **Affect:**
  - 리뷰에서 정서적 단어의 비율​

* **emotion:**
  - 리뷰에서 감정적 단어의 비율​

* **Social:**
  - 리뷰에서 사회적 단어의 비율​

* **Perception:**
  - 리뷰에서 지각적 단어의 비율​

* **motion:**
  - 리뷰에서 동작 단어의 비율​

* **Psychological Distancing:**
  - 리뷰에서 1인칭 대명사의 비율​

* **reputation:**
  - Yelp.com에서 리뷰어가 가진 친구 수​

* **experience:**
  - 리뷰어가 Yelp.com에서 작성한 리뷰 수​

* **rating:**
  - 리뷰어가 매긴 평점

* **review_extremity:**
  - 리뷰 평점의 극단성​

* **depth:**
  - 리뷰에서 사용된 단어 총 개수​

* **text_structure:**
  - 리뷰의 문장당 단어 수​

* **style:**
  - 리뷰에서 현재 시제 또는 미래 시제의 비율​

In [31]:
train.columns

In [32]:
# 숫자형 변수들을 저장

review_num_feature = ['experience', 'depth', 'reputation', 'text_structure', 'style']

train_review_num = train[review_num_feature]
test_review_num = test[review_num_feature]

In [33]:
train_review_num.head()

# **숫자 변수 Column Scaling**

In [34]:
# 변수들을 스케일링함

Minmax = MinMaxScaler()
train_review_scaled = Minmax.fit_transform(train_review_num)
test_review_scaled = Minmax.transform(test_review_num)

train_review_scaled = pd.DataFrame(train_review_scaled, columns=review_num_feature)
test_review_scaled = pd.DataFrame(test_review_scaled, columns=review_num_feature)

## 모델 투입 Num Column 생성

In [35]:
# 평점은 필수로 포함하고, 몇 개의 변수만 선택해서 모델에 투입예정
train_selected = np.array(train_review_scaled)
test_selected = np.array(test_review_scaled)

# **CNN-Biderectional LSTM 모델**
- 비선형 관계를 포착하는 DNN의 특징을 고려하여, 텍스트를 필수로 투입하고 숫자 변수를 선택해서 투입
- 평점의 경우(1~5 사이의 값) 숫자 변수와 같이 투입을 해줌


In [37]:
model = CNN_LSTM_helpful(padded_train_text, train_selected)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(model.summary())

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 300)]                0         []                            
                                                                                                  
 Embedding_Layer (Embedding  (None, 300, 300)             2075700   ['text_input[0][0]']          
 )                                                                                                
                                                                                                  
 Conv1D_3 (Conv1D)           (None, 300, 100)             90100     ['Embedding_Layer[0][0]']     
                                                                                                  
 MaxPooling1D_3 (MaxPooling  (None, 150, 100)             0         ['Conv1D_3[0][0]']        

In [38]:
# early stopping 구현
early_stopping = EarlyStopping(monitor='val_loss', patience=3, min_delta=0.001, restore_best_weights=True)

In [39]:
# 모델 학습
history = model.fit([padded_train_text, train_selected], y_train, validation_split=0.125, epochs=10, batch_size=64, callbacks=[early_stopping])

Epoch 1/10
272/272 [==============================] - 47s 141ms/step - loss: 1.7645 - accuracy: 0.7083 - val_loss: 0.6522 - val_accuracy: 0.7436
Epoch 2/10
272/272 [==============================] - 22s 80ms/step - loss: 0.6112 - accuracy: 0.7405 - val_loss: 0.5945 - val_accuracy: 0.7343
Epoch 3/10
272/272 [==============================] - 15s 54ms/step - loss: 0.5519 - accuracy: 0.7606 - val_loss: 0.5721 - val_accuracy: 0.7415
Epoch 4/10
272/272 [==============================] - 10s 37ms/step - loss: 0.4951 - accuracy: 0.7979 - val_loss: 0.5982 - val_accuracy: 0.7291
Epoch 5/10
272/272 [==============================] - 10s 36ms/step - loss: 0.4345 - accuracy: 0.8339 - val_loss: 0.5963 - val_accuracy: 0.7230
Epoch 6/10
272/272 [==============================] - 8s 28ms/step - loss: 0.3676 - accuracy: 0.8724 - val_loss: 0.6790 - val_accuracy: 0.7174


In [40]:
# 테스트 데이터 예측 및 이진 분류 임계값 적용
y_pred_binary = (model.predict([padded_test_text, test_selected]) >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred_binary)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)

# 모델 성능 평가 지표 계산 및 출력
metrics = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    }

# 성능 지표 출력
for metric, value in metrics.items():
    print(f"{metric}: {value}")


156/156 [==============================] - 2s 7ms/step
Accuracy: 0.7280595813204509
Precision: 0.7673430863614913
Recall: 0.6545893719806763
F1-score: 0.7064957636324136


# **CNN 만 이용**

In [41]:
model = CNN_helpful(padded_train_text, train_selected)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(model.summary())

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 300)]                0         []                            
                                                                                                  
 Embedding_Layer (Embedding  (None, 300, 300)             2075700   ['text_input[0][0]']          
 )                                                                                                
                                                                                                  
 Conv1D_3 (Conv1D)           (None, 300, 100)             90100     ['Embedding_Layer[0][0]']     
                                                                                                  
 GlobalMaxPooling1D_3 (Glob  (None, 100)                  0         ['Conv1D_3[0][0]']      

In [42]:
# 모델 학습
history = model.fit([padded_train_text, train_selected], y_train, validation_split=0.125, epochs=10, batch_size=64, callbacks=[early_stopping])

Epoch 1/10
272/272 [==============================] - 25s 83ms/step - loss: 1.8214 - accuracy: 0.7046 - val_loss: 0.6973 - val_accuracy: 0.7379
Epoch 2/10
272/272 [==============================] - 13s 48ms/step - loss: 0.6364 - accuracy: 0.7345 - val_loss: 0.6166 - val_accuracy: 0.7142
Epoch 3/10
272/272 [==============================] - 10s 38ms/step - loss: 0.5585 - accuracy: 0.7605 - val_loss: 0.5558 - val_accuracy: 0.7480
Epoch 4/10
272/272 [==============================] - 6s 23ms/step - loss: 0.5064 - accuracy: 0.7886 - val_loss: 0.5648 - val_accuracy: 0.7347
Epoch 5/10
272/272 [==============================] - 6s 21ms/step - loss: 0.4336 - accuracy: 0.8362 - val_loss: 0.6235 - val_accuracy: 0.7166
Epoch 6/10
272/272 [==============================] - 6s 21ms/step - loss: 0.3309 - accuracy: 0.8937 - val_loss: 0.7100 - val_accuracy: 0.7230


In [43]:
# 테스트 데이터 예측 및 이진 분류 임계값 적용
y_pred_binary = (model.predict([padded_test_text, test_selected]) >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred_binary)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)

# 모델 성능 평가 지표 계산 및 출력
metrics = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    }

# 성능 지표 출력
for metric, value in metrics.items():
    print(f"{metric}: {value}")

156/156 [==============================] - 0s 2ms/step
Accuracy: 0.7369162640901772
Precision: 0.7364403374849338
Recall: 0.7379227053140096
F1-score: 0.7371807761914337


# **CNN + Bidirectional RNN 사용**

In [44]:
model = CNN_RNN_helpful(padded_train_text, train_selected)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(model.summary())

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text_input (InputLayer)     [(None, 300)]                0         []                            
                                                                                                  
 Embedding_Layer (Embedding  (None, 300, 300)             2075700   ['text_input[0][0]']          
 )                                                                                                
                                                                                                  
 Conv1D_3 (Conv1D)           (None, 300, 100)             90100     ['Embedding_Layer[0][0]']     
                                                                                                  
 MaxPooling1D_3 (MaxPooling  (None, 150, 100)             0         ['Conv1D_3[0][0]']      

In [45]:
# 모델 학습
history = model.fit([padded_train_text, train_selected], y_train, validation_split=0.125, epochs=10, batch_size=64, callbacks=[early_stopping])

Epoch 1/10
272/272 [==============================] - 74s 255ms/step - loss: 1.7975 - accuracy: 0.6626 - val_loss: 0.6824 - val_accuracy: 0.6936
Epoch 2/10
272/272 [==============================] - 61s 223ms/step - loss: 0.6326 - accuracy: 0.7167 - val_loss: 0.5981 - val_accuracy: 0.7271
Epoch 3/10
272/272 [==============================] - 55s 203ms/step - loss: 0.5711 - accuracy: 0.7528 - val_loss: 0.5797 - val_accuracy: 0.7303
Epoch 4/10
272/272 [==============================] - 54s 199ms/step - loss: 0.5152 - accuracy: 0.7908 - val_loss: 0.5965 - val_accuracy: 0.7250
Epoch 5/10
272/272 [==============================] - 53s 197ms/step - loss: 0.4631 - accuracy: 0.8245 - val_loss: 0.6071 - val_accuracy: 0.7198
Epoch 6/10
272/272 [==============================] - 53s 193ms/step - loss: 0.4031 - accuracy: 0.8591 - val_loss: 0.6576 - val_accuracy: 0.7029


In [46]:
# 테스트 데이터 예측 및 이진 분류 임계값 적용
y_pred_binary = (model.predict([padded_test_text, test_selected]) >= 0.5).astype(int)

accuracy = accuracy_score(y_test, y_pred_binary)
precision = precision_score(y_test, y_pred_binary)
recall = recall_score(y_test, y_pred_binary)
f1 = f1_score(y_test, y_pred_binary)

# 모델 성능 평가 지표 계산 및 출력
metrics = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    }

# 성능 지표 출력
for metric, value in metrics.items():
    print(f"{metric}: {value}")

156/156 [==============================] - 4s 21ms/step
Accuracy: 0.7188003220611916
Precision: 0.7438313144908031
Recall: 0.6674718196457327
F1-score: 0.7035858264375132
